.00

<div style="padding:20px; 
            color:#150d0a;
            margin:10px;
            font-size:220%;
            text-align:center;
            display:fill;
            border-radius:20px;
            border-width: 5px;
            border-style: solid;
            border-color: #150d0a;
            background-color:#eca912;
            overflow:hidden;
            font-weight:500">Solar Power Generation prediction using multiple models </div>


<center>
<img src="https://www.itl.cat/pngfile/big/44-448030_download-solar-cell-wallpaper-gallery-solar-pv-module.jpg" width=1000>
</center>

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from functools import cmp_to_key
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
import time

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("📁 Folder:", root)
    for f in files:
        print("   📄", f)

In [ ]:
df = pd.read_csv('/kaggle/input/solar-plant-generation-data/Solar.csv')
df.head()

In [ ]:
df.corr()['SystemProduction'].sort_values(ascending=False).to_frame()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf

decomposed_results = seasonal_decompose(df["SystemProduction"], period=24)

fig, ax = plt.subplots(figsize=(17, 6))
ax.plot(decomposed_results.trend, alpha=0.6, color='Blue', label='Generation Trend', linewidth = 2.0)
ax.plot(decomposed_results.trend.rolling(600).mean().shift(-600), alpha=1, color='Red', label='Generation Trend (Rolling Mean)', linewidth = 5.0)
ax.legend(loc='best')
ax.set_ylabel("Generation Trend")
ax.set_xlabel("Hours")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df["SystemProduction"][:400], alpha=1, color='blue', label='Generation', linewidth = 2.0)
ax.legend(loc='best')
ax.set_ylabel("Generation")
ax.set_xlabel("Hours")
plt.show()

In [ ]:
df_new = df.iloc[0:24 , 0:10]
df_new

In [ ]:
def plot(variable):
    plt.figure(figsize=(15,5))
    plt.plot(df[variable])
    plt.xlabel(variable)
    plt.ylabel("frequency")
    plt.title("{} distribution with plot ".format(variable)) 

In [ ]:
numeric_columns = ['WindSpeed', 'Sunshine', 'AirPressure', 'Radiation',
       'AirTemperature', 'RelativeAirHumidity', 'SystemProduction']
for num in numeric_columns:
    plot(num)
plt.show()

In [ ]:
df.head(1)

In [ ]:
def plot_hist(variable):
    plt.figure(figsize=(15,5))
    sns.histplot(df[variable], kde=True, color='purple')
    plt.xlabel(variable)
    plt.ylabel("frequency")
    plt.axvline(df[variable].mean(), color='r', linestyle='dashed', linewidth=2)
    plt.title("{} distribution with hist ".format(variable))
    plt.show()

In [ ]:
for num in numeric_columns:
    plot_hist(num)

In [ ]:
pd.plotting.register_matplotlib_converters()
from scipy.stats import probplot


def visualize_target(df, target):
    
    print(f'{target}\n{"-" * len(target)}')
        
    print(f'Mean: {df[target].mean():.4f}  -  Median: {df[target].median():.4f}  -  Std: {df[target].std():.4f}')
    print(f'Min: {df[target].min():.4f}  -  25%: {df[target].quantile(0.25):.4f}  -  50%: {df[target].quantile(0.5):.4f}  -  75%: {df[target].quantile(0.75):.4f}  -  Max: {df[target].max():.4f}')
    print(f'Skew: {df[target].skew():.4f}  -  Kurtosis: {df[target].kurtosis():.4f}')
    missing_count = df[df[target].isnull()].shape[0]
    total_count = df.shape[0]
    print(f'Missing Values: {missing_count}/{total_count} ({missing_count * 100 / total_count:.4f}%)')

    fig, axes = plt.subplots(ncols=2, figsize=(24, 6), dpi=100)

    sns.kdeplot(df[target], label=target, fill=True, ax=axes[0])
    axes[0].axvline(df[target].mean(), label='Mean', color='r', linewidth=2, linestyle='--')
    axes[0].axvline(df[target].median(), label='Median', color='b', linewidth=2, linestyle='--')
    axes[0].legend(prop={'size': 15})
    probplot(df[target], plot=axes[1])
    
    for i in range(2):
        axes[i].tick_params(axis='x', labelsize=12.5)
        axes[i].tick_params(axis='y', labelsize=12.5)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')
    axes[0].set_title(f'{target} Distribution', fontsize=15, pad=12)
    axes[1].set_title(f'{target} Probability Plot', fontsize=15, pad=12)
    
    plt.show()

visualize_target(df, 'SystemProduction')

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

In [ ]:
df['Dates'] = pd.to_datetime(df['Date-Hour(NMT)']).dt.date
df['Hour'] = pd.to_datetime(df['Date-Hour(NMT)']).dt.time
df['Day'] = pd.to_datetime(df['Date-Hour(NMT)']).dt.day
df['Month'] = pd.to_datetime(df['Date-Hour(NMT)']).dt.month
df['Year'] = pd.to_datetime(df['Date-Hour(NMT)']).dt.year
df.head()


In [ ]:
KeyError: 'Date-Hour(NMT)'
print(df.columns)

In [ ]:
df["Hour"] = df["Hour"].astype(str)
df['Hour'] = df['Hour'].str.slice(stop=2)
df['Hour'] = df['Hour'].apply(lambda x: x.replace(':', '')).astype(int)

In [ ]:
df["Season"] = [ "Winter" if i < 3 or i > 11 else "Spring" if 3 <= i < 6 else "Summer" if 6 <= i < 9 else "Autumn" for i in df["Month"]]
df.head()

In [ ]:
df_DnN = df.groupby(["Season", "Hour","SystemProduction"], as_index = False)["SystemProduction"].mean().sort_values(by="Hour",ascending = False)

In [ ]:
table = pd.pivot_table(df_DnN, values = "SystemProduction", columns = "Season", index = "Hour", aggfunc = np.mean )
table

In [ ]:
plt.figure(figsize=(17,10))
sns.heatmap(table, annot = True, fmt = ".4f", linewidth = 0.2 ); # 

In [ ]:
df.loc[((df["Hour"] > 7) & (df['Hour'] < 18)) & (df['Season']=='Winter'), 'Time'] = 'Day' 
df.loc[((df["Hour"] > 5) & (df['Hour'] < 19)) & (df['Season']=='Spring'), 'Time'] = 'Day' 
df.loc[((df["Hour"] > 5) & (df['Hour'] < 20)) & (df['Season']=='Summer'), 'Time'] = 'Day' 
df.loc[((df["Hour"] > 6) & (df['Hour'] < 18)) & (df['Season']=='Autumn'), 'Time'] = 'Day' 
df.loc[((df["Hour"] <= 7) | (df['Hour'] >= 18)) & (df['Season']=='Winter'), 'Time'] = 'Night'
df.loc[((df["Hour"] <= 5) | (df['Hour'] >= 19)) & (df['Season']=='Spring'), 'Time'] = 'Night' 
df.loc[((df["Hour"] <= 5) | (df['Hour'] >= 20)) & (df['Season']=='Summer'), 'Time'] = 'Night' 
df.loc[((df["Hour"] <= 6) | (df['Hour'] >= 18)) & (df['Season']=='Autumn'), 'Time'] = 'Night' 

df.head()

In [ ]:
sns.catplot(data = df, x = "Month", y = "RelativeAirHumidity", hue = "Year", kind = "bar", height = 7.5, aspect = 2);

In [ ]:
df_humi = df.groupby(["Radiation", "SystemProduction", "Year", "Season"], as_index = False)["Radiation", "SystemProduction"].mean().sort_values(by="Season", ascending = False)

In [ ]:
table2 = pd.pivot_table(df_humi, values =["Radiation", "SystemProduction"], index = ["Season","Year"])
table2

In [ ]:
df.loc[(df["RelativeAirHumidity"] >= 60.), "Humidity"] = 'Wet' 
df.loc[(df["RelativeAirHumidity"] < 60.), "Humidity"] = 'Dry'
df.head()

In [ ]:
sns.displot(data=df, x="AirTemperature", kde=True, bins = 100,color = "red", facecolor = "#3F7F7F",height = 5, aspect = 3.5);

sns.displot(data=df, x="AirPressure", kde=True, bins = 100,color = "red", facecolor = "Brown",height = 5, aspect = 3.5);

sns.displot(data=df, x="WindSpeed", kde=True, bins = 70,color = "red", facecolor = "Gold",height = 5, aspect = 3.5);

In [ ]:
df = df.drop(['Date-Hour(NMT)'],axis=1)
df.head(1)

In [ ]:
import category_encoders as ce 

encoder = ce.OrdinalEncoder(cols=['Season','Time','Humidity'])
df = encoder.fit_transform(df)
df.head() 

In [ ]:
X = df.drop(['SystemProduction'],axis=1)
y = df['SystemProduction']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=21)

In [ ]:
df['date_year'] = df['Year']
df['date_month'] = df['Month']
df['date_hour'] = df['Day']

df.loc[df.date_month.isin([12,1,2]),"date_season"] = "Autom"
df.loc[df.date_month.isin([3,4,5]), "date_season"] = "Summer"
df.loc[df.date_month.isin([6,7,8]), "date_season"] = "Rain"
df.loc[df.date_month.isin([9,10,11]),"date_season"] = "Winter"

df.head()

In [ ]:
df['date_season'].value_counts()

In [ ]:
# Weekly power Generation
plt.figure(figsize=(20, 5))
plt.title("Daily")
df[4000:4000+7*24].SystemProduction.plot();

In [ ]:
# Monthly power Generation
plt.figure(figsize=(20, 5))
plt.title("Daily")
df[4000:4000+30*24].SystemProduction.plot();

In [ ]:
plt.subplots(1, 2, figsize=(20, 5))
plt.subplot(1, 2, 1)
plt.title("Yearly")
df.SystemProduction.plot()
plt.subplot(1, 2, 2)
plt.title("weekly")
df[4000:4000+7*24].SystemProduction.plot();

In [ ]:
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error
from xgboost import plot_importance
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.float_format', lambda x: '%.8f' % x)

In [ ]:
decomposition = sm.tsa.seasonal_decompose(df["SystemProduction"][:10000], period=24)
fig = decomposition.plot()
fig.set_figwidth(30)
fig.set_figheight(10)

In [ ]:
diff = df["SystemProduction"].diff()
plt.figure(figsize=(50,10))
diff[:10000].plot()
plt.show()

In [ ]:
plt.figure(figsize=(50,10))

df['Radiation'][:10000].plot()
df['SystemProduction'][:10000].plot()
plt.legend(['Radiation', 'SystemProduction'])
plt.show()

In [ ]:
import category_encoders as ce 

encoder = ce.OrdinalEncoder(cols=['date_season'])
df = encoder.fit_transform(df)
df.head() 

In [ ]:
X = df.drop(['SystemProduction'],axis=1)
y = df['SystemProduction']

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=21)                                                

In [ ]:
X_train.head()

In [ ]:
def show_values(axs, orient="v", space=.01):
    def _single(ax):
        if orient == "v":
            for p in ax.patches:
                _x = p.get_x() + p.get_width() / 2
                _y = p.get_y() + p.get_height() + (p.get_height()*0.01)
                value = '{:.1f}'.format(p.get_height())
                ax.text(_x, _y, value, ha="center")
        elif orient == "h":
            for p in ax.patches:
                _x = p.get_x() + p.get_width() + float(space)
                _y = p.get_y() + p.get_height() - (p.get_height()*0.5)
                value = '{:.1f}'.format(p.get_width())
                ax.text(_x, _y, value, ha="left")

    if isinstance(axs, np.ndarray):
        for idx, ax in np.ndenumerate(axs):
            _single(ax)
    else:
        _single(axs)

In [ ]:
df_eda=df.copy()

In [ ]:
plt.figure(figsize=(16,7))
a=sns.barplot(x=df_eda.groupby(["Hour"])["SystemProduction"].sum().index.to_list(),
            y=df_eda.groupby(["Hour"])["SystemProduction"].sum())
show_values(a)
plt.xlabel("Date")
plt.ylabel("Enerji KWH")

plt.xticks(rotation=45)
plt.title("Kwh")
plt.show()

In [ ]:
plt.figure(figsize=(16,9))
a=sns.barplot(x=df_eda.groupby(["Month"])["SystemProduction"].sum().index.to_list(),
            y=df_eda.groupby(["Month"])["SystemProduction"].sum())
show_values(a)
plt.xlabel("Date")
plt.ylabel("Power KWH")

plt.xticks(rotation=45)
plt.title("Kwh")
plt.show()

In [ ]:
def df_analysis(data,x):
    df=data.copy()
    df_analysis = pd.DataFrame(data[x].mean(axis=0), columns = ['mean'])
    df_analysis['min']  = data[x].min(axis=0)
    df_analysis['max']  = data[x].max(axis=0)
    df_analysis['std']  = data[x].std(axis=0)
    df_analysis['med']  = data[x].median(axis=0)
    df_analysis['nunique']  = data[x].nunique(axis=0)

 

    df_analysis['nulls'] = data[x].isnull().sum()
    df_analysis['zeros'] = (data[x] == 0).astype(int).sum(axis=0)
    df_analysis['null_ratio'] = df_analysis['nulls'] / len(df)
    df_analysis['zero_ratio'] = df_analysis['zeros'] / len(df)

 

    return df_analysis

In [ ]:
df_analysis(df,df.columns)

In [ ]:
hour_df = df.copy()
hour_df["Hour"] = df["Hour"]

hour_means = hour_df[["Hour", "SystemProduction"]].groupby("Hour").mean().reset_index().values
hour_stds = hour_df[["Hour", "SystemProduction"]].groupby("Hour").std().reset_index().values

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hour_means[:, 0], hour_means[:, 1], alpha=1, color='blue', label='Mean Generation', linewidth = 5.0)
ax.fill_between(hour_means[:, 0], np.maximum(hour_means[:, 1] - 2*hour_stds[:,1],0), hour_means[:, 1] + 2*hour_stds[:,1], color='#888888', alpha=0.2)
ax.legend(loc='best')
ax.set_ylabel("SystemProduction")
ax.set_xlabel("Hour of the Day")
plt.show()

In [ ]:
df_final = df[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']]
df_final.head()

In [ ]:
numerical = df_final[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']]

fig,axis=plt.subplots(ncols=7,nrows=1,figsize=(15,5))
index=0
axis=axis.flatten()

for col,values in numerical.items():
    sns.boxplot(y=col,data=numerical,color='Brown',ax=axis[index])
    index+=1
plt.tight_layout(pad=0.5,w_pad=0.7,h_pad=5.0)

In [ ]:
upper_limit = df_final[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']].quantile(0.90)
upper_limit
 
lower_limit = df_final[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']].quantile(0.10)
lower_limit

In [ ]:
# Capping --> Winsorization
new_solar_cap = df.copy()
new_solar_cap[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']] = np.where(
    new_solar_cap[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']] > upper_limit, upper_limit,
    np.where(new_solar_cap[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']] < lower_limit, lower_limit,
             new_solar_cap[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']]))
df_final['WindSpeed'].describe()
 
# Comparing
plt.figure(figsize=(16,14))
plt.subplot(2,2,1)
sns.distplot(df_final[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']])
plt.xticks(rotation=90)
plt.subplot(2,2,2)
sns.boxplot(df_final[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']])
plt.xticks(rotation=90)
plt.subplot(2,2,3)
sns.distplot(new_solar_cap[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']])
plt.xticks(rotation=90)
plt.subplot(2,2,4)
sns.boxplot(new_solar_cap[['WindSpeed','Sunshine','AirPressure','Radiation','AirTemperature','RelativeAirHumidity','SystemProduction']]);
plt.xticks(rotation=90);

In [ ]:
X = df_final.drop(['SystemProduction'],axis=1)
y = df_final['SystemProduction']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=21)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
#Linear Regression (OLS)
import statsmodels.api as sm

# build a full model using OLS()
linreg_full_model = sm.OLS(y_train, X_train).fit()

# print the summary output
print(linreg_full_model.summary())

In [ ]:
from sklearn.dummy import DummyRegressor

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.linear_model import GammaRegressor
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import LassoLarsIC
from sklearn.linear_model import LassoLarsCV
from sklearn.linear_model import Lars
from sklearn.linear_model import LarsCV
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import Lasso
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import OrthogonalMatchingPursuitCV
from sklearn.linear_model import PassiveAggressiveRegressor
from sklearn.linear_model import OrthogonalMatchingPursuit
from sklearn.linear_model import LassoLars

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import ExtraTreeRegressor

from sklearn.svm import SVR
from sklearn.svm import NuSVR
from sklearn.svm import LinearSVR

from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.neighbors import KNeighborsRegressor

from sklearn.neural_network import MLPRegressor

import xgboost as xgb

In [ ]:
models = []

names = [
    "LinearRegression",
    "HuberRegressor",
    "RidgeCV",
    "BayesianRidge",
    "Ridge",
    "ElasticNetCV",
    "LassoCV",
    "LassoLarsIC",
    "LassoLarsCV",
    "Lars",
    "LarsCV",
    "TweedieRegressor",
    "ElasticNet",
    "Lasso",
    "OrthogonalMatchingPursuitCV",
    "PassiveAggressiveRegressor",
    "OrthogonalMatchingPursuit",
    "DecisionTreeRegressor",
    "ExtraTreeRegressor",
    "LinearSVR",
    "BaggingRegressor",
    "RandomForestRegressor",
    "GradientBoostingRegressor",
    "ExtraTreesRegressor",
    "AdaBoostRegressor",
    "HistGradientBoostingRegressor",
    "KNeighborsRegressor",
    "MLPRegressor",
    "xgb.XGBRegressor"
]

scores = []

clf = [
    LinearRegression(),
    HuberRegressor(),
    RidgeCV(),
    BayesianRidge(),
    Ridge(),
    ElasticNetCV(),
    LassoCV(),
    LassoLarsIC(criterion='bic', normalize=False),
    LassoLarsCV(),
    Lars(n_nonzero_coefs=1, normalize=False),
    LarsCV(),
    TweedieRegressor(),
    ElasticNet(),
    Lasso(alpha=0.1),
    OrthogonalMatchingPursuitCV(),
    PassiveAggressiveRegressor(),
    OrthogonalMatchingPursuit(),
    DecisionTreeRegressor(),
    ExtraTreeRegressor(),
    LinearSVR(),
    BaggingRegressor(),
    RandomForestRegressor(),
    GradientBoostingRegressor(),
    ExtraTreesRegressor(),
    AdaBoostRegressor(),
    HistGradientBoostingRegressor(),
    KNeighborsRegressor(),
    MLPRegressor(),
    xgb.XGBRegressor(verbosity=0)
]

In [ ]:
%%time
for model in clf:
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    scores.append(score)
 
final_scores = pd.DataFrame(zip(names,scores), columns=['Classifier', 'Accuracy'])

final_scores.sort_values(by='Accuracy',ascending=False).style.background_gradient(cmap="tab10").set_properties(**{
            'font-family': 'Comic Sans MS',
            'color': 'Brown',
            'font-size': '15px'
        })

In [ ]:
p = plt.figure(figsize=(18,20))
p = sns.set_palette('bright')
p = sns.set_context('paper', font_scale=1.8)

p = models=final_scores.sort_values(by='Accuracy',ascending=False)

p = sns.barplot(y= 'Classifier', x= 'Accuracy', data= models,palette='Dark2')

for container in p.containers:
    p.bar_label(container,label_type = 'center',padding = 8,size = 20,color = "White",rotation = 0,
    bbox={"boxstyle": "round", "pad": 0.6, "facecolor": "#a9a9a9", "edgecolor": "white", "alpha": .3})

plt.title('COMPARE THE MODEL',fontsize=20,color='#013220')
plt.xlabel('MODEL',fontsize=20,color='#013220')
plt.ylabel('Model Accuracy',fontsize=20,color='#013220');

In [ ]:
from sklearn import metrics

def print_evaluate(true, predicted, train=True):  
    mae = metrics.mean_absolute_error(true, predicted)
    mse = metrics.mean_squared_error(true, predicted)
    rmse = np.sqrt(metrics.mean_squared_error(true, predicted))
    r2_square = metrics.r2_score(true, predicted)
    if train:
        print("========Training Result=======")
        print('MAE: ', mae)
        print('MSE: ', mse)
        print('RMSE: ', rmse)
        print('R2 Square: ', r2_square)
    elif not train:
        print("=========Testing Result=======")
        print('MAE: ', mae)
        print('MSE: ', mse)
        print('RMSE: ', rmse)
        print('R2 Square: ', r2_square)

In [ ]:
print(f"\033[035m\033[1m")
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print_evaluate(y_train, y_train_pred, train=True)
print_evaluate(y_test, y_test_pred, train=False)


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
gbr = HistGradientBoostingRegressor(learning_rate=0.03,max_depth=5,random_state=42)
gbr.fit(X_train,y_train)
prediction = gbr.predict(X_test)
print(prediction)

In [ ]:
cross_checking = pd.DataFrame({'Actual' : y_test , 'Predicted' : prediction})
cross_checking.head()

In [ ]:
cross_checking['Error'] = cross_checking['Actual'] - cross_checking['Predicted']
cross_checking.head()

In [ ]:
cross_checking_final  = cross_checking[cross_checking['Error'] <= 20]
cross_checking_final.sample(25).style.background_gradient(
        cmap='Dark2').set_properties(**{
            'font-family': 'Times New Roman',
            'color': 'LigntGreen',
            'font-size': '15px'
        })

In [ ]:
print(df.columns)

In [ ]:
X_test

In [ ]:
print(y_pred[:10])


In [ ]:
print(len(y_pred))

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(y_pred[:10])

In [ ]:
submission.to_csv("submission.csv", index=False)

In [ ]:
import os

print(os.listdir("/kaggle/working"))

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

print("R2 Score:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
 conclusion cell
This project predicts solar power generation using regression models.
The model was trained and evaluated using RMSE and R² score.
Final predictions were saved as submission.csv for Kaggle submission.